---------------------
**Kiarash Akbari** 
----------------------

------------------------
[ ADDING ALL DATABASES ]
------------------------ 

In [23]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

files = {
    'patients': 'patients.csv',
    'demographics': 'demographics.csv',
    'lab_results': 'lab_results.csv',
    'medications': 'medications.csv',
    'outcomes': 'outcomes.csv',
    'clinicians': 'clinicians.csv',
    'questionnaires': 'questionnaires.csv',
    'treatments': 'treatments.csv',
    'visits': 'visits.csv'

}

#loading files
for table_name, file_name in files.items():
    try:
        table = pd.read_csv(file_name)
        print(f"\n--- first row of '{table_name}' ---")
        print(table.head(1))
        print("-" * 50)
        table.to_sql(table_name, conn, index=False, if_exists='replace')
    except Exception as e:
        print(f"faild loading {file_name}: {e}")

print('Done')


--- first row of 'patients' ---
   patient_id registered_date
0      100000      2021-04-13
--------------------------------------------------

--- first row of 'demographics' ---
   patient_id  gender  age education  employment   bmi
0      100000  Female   36       BSc  Unemployed  26.5
--------------------------------------------------

--- first row of 'lab_results' ---
   patient_id  lab_test_result              lab_name
0      100000             49.5  DepressionScaleScore
--------------------------------------------------

--- first row of 'medications' ---
   med_id med_name class
0      10   SSRI_A  SSRI
--------------------------------------------------

--- first row of 'outcomes' ---
   patient_id  risk_score risk_level drug_response
0      100000       14.38        Low     Responder
--------------------------------------------------

--- first row of 'clinicians' ---
   clinician_id     name      specialty
0          2000  Dr_2000  ClinicalPsych
---------------------------

-----------------
[ LEVEL 1 - SQL ]
-----------------

In [41]:
query1_1 = """
SELECT
    COUNT(p.patient_id) AS  null_bmi_count
FROM 
    Patients p
JOIN
    demographics d ON p.patient_id = d.patient_id
WHERE
    d.bmi IS NULL;
"""

resultQ1 = pd.read_sql(query1_1, conn)
print(resultQ1)

   null_bmi_count
0             200


In [42]:
query1_2 = """
SELECT 
    m.class,
    AVG(t.baseline_severity) AS avg_baseline_severity
FROM
    treatments t
JOIN
    medications m ON t.med_id = m.med_id
GROUP BY
    m.class;
"""

resultQ2 = pd.read_sql(query1_2, conn)
print(resultQ2)

            class  avg_baseline_severity
0         Adjunct               6.982730
1  MoodStabilizer               6.894381
2            SNRI               6.845751
3            SSRI               6.986462


In [43]:
query1_3 = """
SELECT 
    c.name,
    c.specialty,
    COUNT(*) as visit_count
FROM visits v
JOIN clinicians c ON v.clinician_id = c.clinician_id
GROUP BY c.clinician_id, c.name, c.specialty
ORDER BY visit_count DESC
LIMIT 5;
"""

try:
    resultQ3 = pd.read_sql(query1_3, conn)
    print(f"top 5 clinicians: \n{resultQ3}")
except Exception as e:
    print(f"error in {e}")

top 5 clinicians: 
      name      specialty  visit_count
0  Dr_2006  ClinicalPsych          441
1  Dr_2009  ClinicalPsych          431
2  Dr_2001  ClinicalPsych          420
3  Dr_2011     Counseling          419
4  Dr_2012     Counseling          408


In [44]:
query1_4 = """
SELECT
    visit_type,
    COUNT(*) as count
FROM visits
GROUP BY visit_type
ORDER BY count DESC;
"""

try:
    resultQ4 = pd.read_sql(query1_4, conn)
    print(f"visti types: \n {resultQ4}")
except Exception as e:
    print(f"error in {e}")

visti types: 
      visit_type  count
0    outpatient   4472
1  telemedicine   1231
2     inpatient    280


-----------------
[ LEVEL 2 - SQL ]
-----------------

In [59]:
query2_1 = """
SELECT
    patient_id,
    adherence_pct
FROM treatments
WHERE adherence_pct < 50
    OR adherence_pct > 100
    OR adherence_pct < 0;
"""

try:
    resultQ2_1 = pd.read_sql(query2_1, conn)
    print(f"patients with low or inconsistent adherence: ")
    print(f"\n {resultQ2_1}")
except Exception as e:
    print(f"error in {e}")

patients with low or inconsistent adherence: 

       patient_id  adherence_pct
0         100002      21.285627
1         100007      29.888373
2         100018      25.667007
3         100019      38.759244
4         100021      46.308267
...          ...            ...
1501      104979      24.907185
1502      104986      26.489197
1503      104990      42.371759
1504      104993      35.369114
1505      104998      21.320869

[1506 rows x 2 columns]


In [62]:
query2_2_avg_risk = """ 
SELECT 
    d.education, 
    AVG(o.risk_score) as avg_risk_score
FROM demographics d
JOIN outcomes o ON d.patient_id = o.patient_id
GROUP BY d.education
ORDER BY avg_risk_score DESC;
"""

query2_2_risk_dist = """ 
SELECT
    d.education,
    o.risk_level,
    COUNT(*) as count
FROM demographics d
JOIN outcomes o ON d.patient_id = o.patient_id
GROUP BY d.education, o.risk_level
ORDER BY d.education, o.risk_level;
"""

try:
    table_avg = pd.read_sql(query2_2_avg_risk, conn)
    print("average risk score:")
    print(f"\n {table_avg}")

    table_dist = pd.read_sql(query2_2_risk_dist, conn)
    print("\nrisk level distirbution:")
    print(f"\n {table_dist}")
except Exception as e:
    print(f"error in {e}")

average risk score:

     education  avg_risk_score
0        NoHS       24.368517
1         BSc       24.100924
2         PhD       24.075257
3  HighSchool       23.991284
4         MSc       23.941075

risk level distirbution:

      education risk_level  count
0          BSc       High      7
1          BSc        Low   1030
2          BSc     Medium    706
3   HighSchool       High      7
4   HighSchool        Low   1074
5   HighSchool     Medium    734
6          MSc       High      7
7          MSc        Low    447
8          MSc     Medium    290
9         NoHS       High      2
10        NoHS        Low    249
11        NoHS     Medium    194
12         PhD       High      1
13         PhD        Low    138
14         PhD     Medium    114


----------------------------------
[ LEVEL 3 - DATA PREPARATION ]
----------------------------------

In [65]:
#load datasets
patients = pd.read_csv('patients.csv')
demographics = pd.read_csv('demographics.csv')
outcomes = pd.read_csv('outcomes.csv')
treatments = pd.read_csv('treatments.csv')
lab_results = pd.read_csv('lab_results.csv')
questionnaires= pd.read_csv('questionnaires.csv')
medications = pd.read_csv('medications.csv')